# PesaPap Customer Growth, Retention and Value Analytics

## Notebook 1: Data Cleaning and Quality Audit

This notebook generates the synthetic capstone data, audits six dimensions of data quality, applies the reusable cleaning function in `src/cleaning.py`, saves clean tables, and loads them into SQLite for later SQL analysis.


In [1]:
from pathlib import Path
import sys
import subprocess
import sqlite3
import pandas as pd
import numpy as np

# Detect project root in local/Jupyter/Colab environments
cwd = Path.cwd()

if (cwd / "data").exists() and (cwd / "src").exists():
    ROOT = cwd
elif (cwd.parent / "data").exists() and (cwd.parent / "src").exists():
    ROOT = cwd.parent
elif Path("/content/drive/MyDrive/plp-capstone-B-robin-kinyua").exists():
    ROOT = Path("/content/drive/MyDrive/plp-capstone-B-robin-kinyua")
else:
    raise FileNotFoundError(
        "Project root not found. Run the notebook from the repository root "
        "or the notebooks folder."
    )

DATA = ROOT / "data"
PROCESSED = DATA / "processed"
SRC = ROOT / "src"

PROCESSED.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)


Project root: /content/drive/MyDrive/plp-capstone-B-robin-kinyua


## 1. Generate Raw Data

The project uses synthetic fintech data generated with a fixed random seed for reproducibility. The generator intentionally introduces a small number of controlled quality issues so the cleaning workflow can be demonstrated.


In [2]:
subprocess.run(
    [sys.executable, str(DATA / "generate_capstone_data.py")],
    cwd=ROOT,
    check=True
)

users = pd.read_csv(DATA / "users.csv")
transactions = pd.read_csv(DATA / "transactions.csv")
campaigns = pd.read_csv(DATA / "campaigns.csv")
activity = pd.read_csv(DATA / "activity.csv")

print("Users:", users.shape)
print("Transactions:", transactions.shape)
print("Campaigns:", campaigns.shape)
print("Activity:", activity.shape)


Users: (8000, 5)
Transactions: (70828, 6)
Campaigns: (12, 5)
Activity: (8000, 4)


## 2. Six-Dimension Data Quality Audit

The audit checks:

1. **Completeness**: missing values
2. **Uniqueness**: duplicate keys or rows
3. **Validity**: values outside expected rules
4. **Consistency**: inconsistent text formats or types
5. **Accuracy**: obvious logical or numerical problems
6. **Timeliness**: dates outside the intended analysis period


In [3]:
audit = []

audit.append({
    "dimension": "Completeness",
    "check": "Missing values",
    "result": transactions.isna().sum().to_dict()
})

audit.append({
    "dimension": "Uniqueness",
    "check": "Duplicate transaction IDs",
    "result": int(transactions["txn_id"].duplicated().sum())
})

audit.append({
    "dimension": "Validity",
    "check": "Non-positive transaction amounts",
    "result": int((transactions["amount"] <= 0).sum())
})

audit.append({
    "dimension": "Consistency",
    "check": "Raw product and channel labels",
    "result": {
        "products": sorted(transactions["product"].dropna().unique().tolist()),
        "channels": sorted(transactions["channel"].dropna().unique().tolist())
    }
})

users_check = users.copy()
tx_check = transactions.copy()

users_check["signup_date"] = pd.to_datetime(
    users_check["signup_date"],
    errors="coerce"
)

tx_check["txn_date"] = pd.to_datetime(
    tx_check["txn_date"],
    errors="coerce"
)

lifecycle = tx_check.merge(
    users_check[["user_id", "signup_date"]],
    on="user_id",
    how="left"
)

audit.append({
    "dimension": "Accuracy",
    "check": "Transactions before signup",
    "result": int(
        (lifecycle["txn_date"] < lifecycle["signup_date"]).sum()
    )
})

analysis_start = pd.Timestamp("2025-07-01")
analysis_end = pd.Timestamp("2026-06-30")

audit.append({
    "dimension": "Timeliness",
    "check": "Transactions outside analysis period",
    "result": int(
        (
            (tx_check["txn_date"] < analysis_start) |
            (tx_check["txn_date"] > analysis_end)
        ).sum()
    )
})

for item in audit:
    print("\n", item["dimension"])
    print(item["check"])
    print(item["result"])



 Completeness
Missing values
{'txn_id': 0, 'user_id': 0, 'txn_date': 0, 'product': 0, 'amount': 1, 'channel': 0}

 Uniqueness
Duplicate transaction IDs
2

 Validity
Non-positive transaction amounts
1

 Consistency
Raw product and channel labels
{'products': [' send money ', 'Airtime', 'Bill Pay', 'Merchant Pay', 'Savings', 'Send Money'], 'channels': ['Agent', 'App', 'USSD', 'app']}

 Accuracy
Transactions before signup
1

 Timeliness
Transactions outside analysis period
0


## 3. Apply Reusable Cleaning Function

The reusable cleaning logic is stored in `src/cleaning.py`. It standardises text, removes duplicates and invalid records, imputes the missing amount using the positive median, and returns both cleaned tables and a cleaning log.


In [4]:
sys.path.insert(0, str(SRC))

from cleaning import clean_capstone_data

clean_tables, cleaning_log = clean_capstone_data(
    users,
    transactions,
    campaigns,
    activity
)

display(cleaning_log)


,table,issue,action,rows_affected
0,users,Invalid or missing signup dates,Converted signup_date to datetime,0
1,transactions,Duplicate transaction IDs,Removed duplicate txn_id rows,2
2,transactions,Inconsistent product text,Trimmed whitespace and standardised product case,1
3,transactions,Inconsistent channel text,Standardised channel case,1
4,transactions,Missing transaction amount,Imputed with positive median amount (988),1
5,transactions,Non-positive transaction amount,Removed rows with amount <= 0,1
6,transactions,Invalid transaction dates,Parsed dates and removed unparseable rows,0
7,transactions,Transactions before user signup,Removed lifecycle-invalid transactions,1
8,campaigns,Campaign date type,Converted campaign_month to datetime,0
9,activity,Activity field consistency,Standardised numeric fields and recalculated a...,0


## 4. Recalculate Customer Activity

The activity table is recalculated from the final cleaned transactions so transaction counts, total value and active status remain consistent with the cleaned data.


In [5]:
clean_users = clean_tables["users"]
clean_transactions = clean_tables["transactions"]

clean_activity = (
    clean_transactions
    .groupby("user_id")
    .agg(
        transaction_count=("txn_id", "count"),
        total_value=("amount", "sum")
    )
    .reset_index()
)

clean_activity = (
    clean_users[["user_id"]]
    .merge(clean_activity, on="user_id", how="left")
)

clean_activity["transaction_count"] = (
    clean_activity["transaction_count"]
    .fillna(0)
    .astype(int)
)

clean_activity["total_value"] = (
    clean_activity["total_value"]
    .fillna(0)
)

clean_activity["active_flag"] = (
    clean_activity["transaction_count"] > 0
).astype(int)

clean_tables["activity"] = clean_activity


## 5. Save Clean Tables


In [6]:
clean_tables["users"].to_csv(
    PROCESSED / "users_clean.csv",
    index=False
)

clean_tables["transactions"].to_csv(
    PROCESSED / "transactions_clean.csv",
    index=False
)

clean_tables["campaigns"].to_csv(
    PROCESSED / "campaigns_clean.csv",
    index=False
)

clean_tables["activity"].to_csv(
    PROCESSED / "activity_clean.csv",
    index=False
)

cleaning_log.to_csv(
    PROCESSED / "cleaning_log.csv",
    index=False
)

print("Clean files saved.")


Clean files saved.


## 6. Load Clean Tables into SQLite


In [7]:
db_path = DATA / "pesapap_capstone.db"

conn = sqlite3.connect(db_path)

clean_tables["users"].to_sql(
    "users",
    conn,
    if_exists="replace",
    index=False
)

clean_tables["transactions"].to_sql(
    "transactions",
    conn,
    if_exists="replace",
    index=False
)

clean_tables["campaigns"].to_sql(
    "campaigns",
    conn,
    if_exists="replace",
    index=False
)

clean_tables["activity"].to_sql(
    "activity",
    conn,
    if_exists="replace",
    index=False
)

for table in ["users", "transactions", "campaigns", "activity"]:
    result = pd.read_sql_query(
        f"SELECT COUNT(*) AS rows FROM {table};",
        conn
    )
    print(table, int(result.loc[0, "rows"]))

conn.close()


users 8000
transactions 70824
campaigns 12
activity 8000


## 7. Final Validation

The final cleaned transaction table should contain **70,824 unique valid transactions** with a total transaction value of **KES 111,178,105**.


In [8]:
print("FINAL VALIDATION")
print("Users:", clean_tables["users"].shape)
print("Transactions:", clean_tables["transactions"].shape)
print("Campaigns:", clean_tables["campaigns"].shape)
print("Activity:", clean_tables["activity"].shape)

print(
    "Unique transaction IDs:",
    clean_tables["transactions"]["txn_id"].nunique()
)

print(
    "Total transaction value:",
    clean_tables["transactions"]["amount"].sum()
)

print(
    "Average transaction value:",
    clean_tables["transactions"]["amount"].mean()
)

print(
    "Active customers:",
    clean_tables["activity"]["active_flag"].sum()
)


FINAL VALIDATION
Users: (8000, 5)
Transactions: (70824, 7)
Campaigns: (12, 5)
Activity: (8000, 4)
Unique transaction IDs: 70824
Total transaction value: 111178105.0
Average transaction value: 1569.7800886705072
Active customers: 7522


## Cleaning Summary

The raw transaction data contained controlled quality issues including two duplicate transaction IDs, one missing transaction amount, one non-positive amount, inconsistent product and channel labels, and one transaction dated before the customer's signup date.

After cleaning, the project contains 70,824 unique valid transactions. The cleaned transaction value is KES 111,178,105 and 7,522 of the 8,000 registered users completed at least one valid transaction.

These clean tables are used consistently throughout the exploratory, SQL, statistical and dashboard analyses.
